# VKR Defense Training Notebook

Этот ноутбук - тренажер для подготовки к защите магистерской диссертации по проекту классификации и подсчета `cattle` / `sheep`.

Как работать:

1. Запускай ячейки сверху вниз.
2. Сначала отвечай своими словами в переменных `my_answer_*`.
3. Потом сравнивай с эталонной формулировкой.
4. Если ответ на русском получается уверенно, проговори короткую English viva version.

Важно: в текущем архиве реализованы notebooks 01-05. Поэтому классификационные метрики можно заявлять, а YOLOv8/counting metrics нужно называть планируемыми, если они еще не получены отдельно.

In [13]:
from pprint import pprint
import random
import math
import statistics

PROJECT_FACTS = {
    "raw_images_total": 17759,
    "waid_images": 14366,
    "coco_images": 3393,
    "filtered_images": 12232,
    "train_images": 8562,
    "val_images": 1835,
    "test_images": 1835,
    "crops_total": 51320,
    "train_crops": 35806,
    "val_crops": 7770,
    "test_crops": 7744,
    "cattle_crops": 26986,
    "sheep_crops": 24334,
    "custom_cnn_acc": 0.9610,
    "resnet50_acc": 0.9570,
    "vgg16_acc": 0.9738,
    "mobilenet_v2_acc": 0.9498,
    "custom_cnn_f1": 0.9608,
    "vgg16_f1": 0.9736,
    "augmentation_delta": 0.0062,
}

pprint(PROJECT_FACTS)

{'augmentation_delta': 0.0062,
 'cattle_crops': 26986,
 'coco_images': 3393,
 'crops_total': 51320,
 'custom_cnn_acc': 0.961,
 'custom_cnn_f1': 0.9608,
 'filtered_images': 12232,
 'mobilenet_v2_acc': 0.9498,
 'raw_images_total': 17759,
 'resnet50_acc': 0.957,
 'sheep_crops': 24334,
 'test_crops': 7744,
 'test_images': 1835,
 'train_crops': 35806,
 'train_images': 8562,
 'val_crops': 7770,
 'val_images': 1835,
 'vgg16_acc': 0.9738,
 'vgg16_f1': 0.9736,
 'waid_images': 14366}


## 1. Flashcards: быстрые вопросы

Запусти код ниже. Используй `show_card(i)` чтобы увидеть вопрос, затем сначала ответь устно, и только потом вызови `reveal(i)`.

In [23]:
FLASHCARDS = [
    {
        "q_ru": "В чем тема работы?",
        "a_ru": "Автоматическая классификация и последующий подсчет сельскохозяйственных животных на изображениях, с фокусом на cattle и sheep.",
        "a_en": "The thesis focuses on automatic livestock classification and the next step of animal counting, mainly cattle and sheep.",
        "must_say": "2 класса; classification готова; YOLO/counting как следующий этап.",
    },
    {
        "q_ru": "Какие данные использовались?",
        "a_ru": "WAID с дроновыми изображениями и COCO 2017 cow/sheep subset с наземными изображениями.",
        "a_en": "I used WAID drone images and a COCO 2017 cow/sheep ground-level subset.",
        "must_say": "14 366 WAID + 3 393 COCO = 17 759 raw images.",
    },
    {
        "q_ru": "Почему использовались crops, а не полные изображения?",
        "a_ru": "В полном кадре много фона, особенно в WAID, поэтому классификатор мог бы выучить shortcut. Crops заставляют модель смотреть на животное.",
        "a_en": "Full images contain too much background, so crops force the classifier to focus on the animal.",
        "must_say": "51 320 crops; full images важны для detection, но не для crop classification.",
    },
    {
        "q_ru": "Что было baseline?",
        "a_ru": "Собственная CNN, обученная с нуля и настроенная через Optuna.",
        "a_en": "The baseline was a custom CNN trained from scratch and tuned with Optuna.",
        "must_say": "Custom CNN test accuracy 0.9610.",
    },
    {
        "q_ru": "Какая модель лучшая?",
        "a_ru": "VGG16: test accuracy 0.9738 и macro F1 0.9736.",
        "a_en": "VGG16 performed best with 0.9738 test accuracy and 0.9736 macro F1.",
        "must_say": "Не обобщать на все задачи; это результат данного setup.",
    },
    {
        "q_ru": "Почему VGG16 могла обогнать ResNet50?",
        "a_ru": "В этом setup VGG16 имела гораздо более крупную trainable classification head: около 3.2M параметров против 0.26M у ResNet50.",
        "a_en": "VGG16 had a much larger trainable head in this setup, so it had more adaptation capacity.",
        "must_say": "Это не значит, что VGG16 всегда лучше ResNet50.",
    },
    {
        "q_ru": "Почему accuracy недостаточно?",
        "a_ru": "Accuracy не показывает типы ошибок и может скрывать class-specific failures. Поэтому использовались macro precision, recall и F1.",
        "a_en": "Accuracy can hide class-specific failures, so I also used macro precision, recall, and F1.",
        "must_say": "Macro metrics учитывают оба класса.",
    },
    {
        "q_ru": "Что такое data leakage в crop pipeline?",
        "a_ru": "Если сначала нарезать crops, а потом делить их на train/test, crops из одного изображения могут оказаться в разных split. Поэтому split делался на уровне исходных изображений.",
        "a_en": "Splitting after crop extraction can put crops from the same image into both train and test, so splitting must happen at image level.",
        "must_say": "Image-level split before crop generation.",
    },
    {
        "q_ru": "Что показал Grad-CAM?",
        "a_ru": "Он визуально показал, что модели чаще фокусируются на теле животного, а не только на фоне.",
        "a_en": "Grad-CAM showed that the models mostly focus on the animal body, not only on the background.",
        "must_say": "Grad-CAM - evidence, но не strict proof.",
    },
    {
        "q_ru": "Можно ли уже заявлять YOLOv8 counting metrics?",
        "a_ru": "Нет, в текущих ноутбуках 01-05 есть результаты classification. YOLOv8/counting metrics нужно считать отдельно.",
        "a_en": "No. The current notebooks contain classification results; YOLOv8 counting metrics must be computed separately.",
        "must_say": "Не выдумывать mAP, MAE, R².",
    },
]


def show_card(i=None):
    if i is None:
        i = random.randrange(len(FLASHCARDS))
    card = FLASHCARDS[i]
    print(f"Card #{i}")
    print("Question RU:", card["q_ru"])
    print("Now answer aloud before calling reveal(i).")
    return i


def reveal(i):
    card = FLASHCARDS[i]
    print("Answer RU:", card["a_ru"])
    print("English:", card["a_en"])
    print("Must say:", card["must_say"])

idx = show_card()

Card #8
Question RU: Что показал Grad-CAM?
Now answer aloud before calling reveal(i).


## 2. Pitch exercise

Сначала заполни ответы своими словами. Потом сравни с эталоном.

In [ ]:
my_answer_30s_ru = """
Напиши здесь свой 30-секундный ответ по-русски.
"""

my_answer_30s_en = """
Write your 30-second English version here.
"""

reference_30s_ru = """
Моя работа посвящена автоматической классификации и последующему подсчету сельскохозяйственных животных,
в первую очередь коров и овец. Я объединила WAID и COCO, провела EDA и preprocessing, подготовила 51 320 crops,
обучила собственную CNN baseline и сравнила ее с ResNet50, VGG16 и MobileNetV2. Лучший classification результат
показала VGG16: test accuracy 0.9738, а Custom CNN дала сильный baseline 0.9610.
"""

reference_30s_en = """
My thesis focuses on automatic livestock classification and the next step of animal counting.
I combined WAID and COCO, performed EDA and preprocessing, prepared 51,320 crops, trained a custom CNN baseline,
and compared it with ResNet50, VGG16, and MobileNetV2. VGG16 achieved the best classification accuracy, 0.9738,
while the custom CNN reached a strong 0.9610 baseline.
"""

print("Checklist for your pitch:")
for item in [
    "names the task: livestock classification/counting",
    "mentions cattle/sheep",
    "mentions WAID + COCO",
    "mentions crops, not only full images",
    "mentions Custom CNN baseline",
    "mentions best model and exact accuracy",
    "does not claim YOLO/counting metrics yet",
]:
    print("-", item)

print("\nReference RU:", reference_30s_ru)
print("Reference EN:", reference_30s_en)

Checklist for your pitch:
- names the task: livestock classification/counting
- mentions cattle/sheep
- mentions WAID + COCO
- mentions crops, not only full images
- mentions Custom CNN baseline
- mentions best model and exact accuracy
- does not claim YOLO/counting metrics yet

Reference RU: 
Моя работа посвящена автоматической классификации и последующему подсчету сельскохозяйственных животных,
в первую очередь коров и овец. Я объединила WAID и COCO, провела EDA и preprocessing, подготовила 51 320 crops,
обучила собственную CNN baseline и сравнила ее с ResNet50, VGG16 и MobileNetV2. Лучший classification результат
показала VGG16: test accuracy 0.9738, а Custom CNN дала сильный baseline 0.9610.

Reference EN: 
My thesis focuses on automatic livestock classification and the next step of animal counting.
I combined WAID and COCO, performed EDA and preprocessing, prepared 51,320 crops, trained a custom CNN baseline,
and compared it with ResNet50, VGG16, and MobileNetV2. VGG16 achieved th

## 3. Metrics from scratch: accuracy, precision, recall, F1

Toy task: `0 = cattle`, `1 = sheep`. Сначала попробуй вручную посчитать confusion matrix, потом запусти код.

In [16]:
y_true = [0, 0, 0, 0, 1, 1, 1, 1, 1, 0]
y_pred = [0, 0, 1, 0, 1, 1, 0, 1, 1, 0]
class_names = {0: "cattle", 1: "sheep"}


def confusion_matrix_binary(y_true, y_pred):
    cm = [[0, 0], [0, 0]]
    for t, p in zip(y_true, y_pred):
        cm[t][p] += 1
    return cm


def per_class_metrics(cm, cls):
    tp = cm[cls][cls]
    fp = sum(cm[r][cls] for r in range(2) if r != cls)
    fn = sum(cm[cls][c] for c in range(2) if c != cls)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

cm = confusion_matrix_binary(y_true, y_pred)
correct = sum(t == p for t, p in zip(y_true, y_pred))
accuracy = correct / len(y_true)

print("Confusion matrix rows=true, cols=pred:")
print("          pred cattle  pred sheep")
print("true cattle", cm[0])
print("true sheep ", cm[1])
print("accuracy:", round(accuracy, 3))

macro = []
for cls in [0, 1]:
    p, r, f1 = per_class_metrics(cm, cls)
    macro.append((p, r, f1))
    print(f"{class_names[cls]}: precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}")

macro_precision = statistics.mean(x[0] for x in macro)
macro_recall = statistics.mean(x[1] for x in macro)
macro_f1 = statistics.mean(x[2] for x in macro)
print("macro precision:", round(macro_precision, 3))
print("macro recall:   ", round(macro_recall, 3))
print("macro f1:       ", round(macro_f1, 3))

Confusion matrix rows=true, cols=pred:
          pred cattle  pred sheep
true cattle [4, 1]
true sheep  [1, 4]
accuracy: 0.8
cattle: precision=0.800, recall=0.800, f1=0.800
sheep: precision=0.800, recall=0.800, f1=0.800
macro precision: 0.8
macro recall:    0.8
macro f1:        0.8


### Самопроверка по метрикам

Ответь устно:

1. Почему при imbalance accuracy может обманывать?
2. Чем precision отличается от recall?
3. Почему macro F1 подходит для защиты этой работы?

Эталон: accuracy может быть высокой из-за majority class; precision отвечает "насколько чисты предсказания класса", recall отвечает "сколько реальных объектов класса найдено"; macro F1 усредняет классы равноправно.

## 4. Counting metrics: MAE and R²

Это не результаты проекта, а игрушечный пример, чтобы быстро понять, как будут оцениваться YOLOv8 counting predictions.

In [17]:
gt_counts = [3, 8, 12, 1, 5, 10]
pred_counts = [2, 9, 10, 1, 7, 8]


def mae(y, yhat):
    return sum(abs(a - b) for a, b in zip(y, yhat)) / len(y)


def r2_score_simple(y, yhat):
    mean_y = sum(y) / len(y)
    ss_res = sum((a - b) ** 2 for a, b in zip(y, yhat))
    ss_tot = sum((a - mean_y) ** 2 for a in y)
    return 1 - ss_res / ss_tot if ss_tot else float("nan")

print("gt_counts:  ", gt_counts)
print("pred_counts:", pred_counts)
print("MAE:", round(mae(gt_counts, pred_counts), 3))
print("R²: ", round(r2_score_simple(gt_counts, pred_counts), 3))

print("\nDefense sentence:")
print("For counting, MAE gives the average absolute count error per image, while R² shows how well predicted counts follow the variation in true counts.")

gt_counts:   [3, 8, 12, 1, 5, 10]
pred_counts: [2, 9, 10, 1, 7, 8]
MAE: 1.333
R²:  0.844

Defense sentence:
For counting, MAE gives the average absolute count error per image, while R² shows how well predicted counts follow the variation in true counts.


### Вопрос для устной тренировки

Комиссия: "Почему classification accuracy не доказывает качество counting?"

Твой ответ должен содержать три части:

1. Classification crop уже предполагает, что объект найден.
2. Counting требует найти все объекты в полном изображении.
3. Поэтому нужны detection metrics и count metrics: mAP, MAE, R².

## 5. McNemar test intuition

McNemar test нужен, когда две модели проверяются на одних и тех же test samples. Он смотрит не просто на две accuracy, а на paired disagreements: где первая модель права, а вторая ошибается, и наоборот.

In [18]:
# 1 = prediction correct, 0 = prediction wrong for the same test samples
model_a_correct = [1, 1, 1, 0, 1, 0, 1, 1, 0, 1]
model_b_correct = [1, 0, 1, 0, 1, 1, 1, 0, 0, 1]

b = 0  # A correct, B wrong
c = 0  # A wrong, B correct
for a, b_model in zip(model_a_correct, model_b_correct):
    if a == 1 and b_model == 0:
        b += 1
    elif a == 0 and b_model == 1:
        c += 1

print("A correct, B wrong:", b)
print("A wrong, B correct:", c)
print("McNemar focuses on these off-diagonal disagreement counts.")

if b + c > 0:
    chi2_continuity = (abs(b - c) - 1) ** 2 / (b + c)
    print("Approx McNemar chi-square with continuity correction:", round(chi2_continuity, 3))

print("\nDefense sentence:")
print("McNemar is appropriate because both models are evaluated on the same test objects, so their errors are paired rather than independent.")

A correct, B wrong: 2
A wrong, B correct: 1
McNemar focuses on these off-diagonal disagreement counts.
Approx McNemar chi-square with continuity correction: 0.0

Defense sentence:
McNemar is appropriate because both models are evaluated on the same test objects, so their errors are paired rather than independent.


## 6. Data leakage exercise: crop-level split vs image-level split

Главная мысль: если делить после crop extraction, один исходный image_id может попасть и в train, и в test.

In [19]:
random.seed(42)

# Toy dataset: 8 images, each image has 3 crops.
crops = []
for image_id in range(8):
    cls = "cattle" if image_id < 4 else "sheep"
    for crop_id in range(3):
        crops.append({"image_id": image_id, "crop_id": crop_id, "class": cls})

# Bad split: shuffle crops directly.
shuffled = crops[:]
random.shuffle(shuffled)
bad_train = shuffled[:14]
bad_test = shuffled[14:]

bad_train_images = {row["image_id"] for row in bad_train}
bad_test_images = {row["image_id"] for row in bad_test}
leaked_images = bad_train_images & bad_test_images

print("BAD crop-level split")
print("train image ids:", sorted(bad_train_images))
print("test image ids: ", sorted(bad_test_images))
print("leaked image ids:", sorted(leaked_images))

# Good split: split image IDs first, then collect crops.
all_images = list(range(8))
random.shuffle(all_images)
train_image_ids = set(all_images[:5])
test_image_ids = set(all_images[5:])

good_train = [row for row in crops if row["image_id"] in train_image_ids]
good_test = [row for row in crops if row["image_id"] in test_image_ids]

good_leak = {row["image_id"] for row in good_train} & {row["image_id"] for row in good_test}

print("\nGOOD image-level split")
print("train image ids:", sorted(train_image_ids))
print("test image ids: ", sorted(test_image_ids))
print("leaked image ids:", sorted(good_leak))

assert len(good_leak) == 0

BAD crop-level split
train image ids: [0, 1, 2, 3, 4, 5, 6, 7]
test image ids:  [0, 1, 2, 3, 6, 7]
leaked image ids: [0, 1, 2, 3, 6, 7]

GOOD image-level split
train image ids: [1, 2, 3, 5, 7]
test image ids:  [0, 4, 6]
leaked image ids: []


### Defense answer template: data leakage

Русский:

> Главный риск leakage был из-за crop generation. Если сначала нарезать crops, а потом делить их случайно, crops из одного изображения могут оказаться в train и test. Поэтому я делала split на уровне исходных изображений, а crops создавались внутри каждого split.

English:

> The main leakage risk came from crop generation. If crops are split randomly after extraction, crops from the same image can appear in both train and test. I avoided this by splitting at the original image level first.

## 7. Stratified split exercise

Stratification сохраняет пропорции классов в split. Ниже игрушечный пример без sklearn.

In [20]:
random.seed(7)
images = []
for i in range(20):
    cls = "cattle" if i < 12 else "sheep"
    images.append({"image_id": i, "class": cls})


def stratified_split(items, train_ratio=0.7):
    by_class = {}
    for item in items:
        by_class.setdefault(item["class"], []).append(item)
    train, test = [], []
    for cls, rows in by_class.items():
        rows = rows[:]
        random.shuffle(rows)
        n_train = round(len(rows) * train_ratio)
        train.extend(rows[:n_train])
        test.extend(rows[n_train:])
    return train, test

train, test = stratified_split(images)


def counts(rows):
    out = {}
    for r in rows:
        out[r["class"]] = out.get(r["class"], 0) + 1
    return out

print("all:  ", counts(images))
print("train:", counts(train))
print("test: ", counts(test))

print("\nDefense sentence:")
print("Stratification keeps cattle/sheep proportions similar across train, validation, and test, so evaluation is not distorted by class imbalance.")

all:   {'cattle': 12, 'sheep': 8}
train: {'cattle': 8, 'sheep': 6}
test:  {'cattle': 4, 'sheep': 2}

Defense sentence:
Stratification keeps cattle/sheep proportions similar across train, validation, and test, so evaluation is not distorted by class imbalance.


## 8. Grad-CAM interpretation drill

Grad-CAM не доказывает модель математически, но помогает увидеть, какие области изображения влияли на решение.

In [21]:
scenarios = [
    {
        "observation": "Heatmap is concentrated on the animal body and head.",
        "interpretation": "Good sign: the model uses relevant animal features.",
    },
    {
        "observation": "Heatmap is mostly on grass or sky, not on the animal.",
        "interpretation": "Possible shortcut: the model may rely on background context.",
    },
    {
        "observation": "Heatmap covers only a corner of the crop.",
        "interpretation": "The prediction may be unstable; inspect the crop and prediction confidence.",
    },
]

for i, s in enumerate(scenarios, 1):
    print(f"Scenario {i}")
    print("Observation:   ", s["observation"])
    print("Interpretation:", s["interpretation"])
    print()

print("Defense sentence:")
print("Grad-CAM is supporting interpretability evidence: it suggests whether the model focuses on animal features or on dataset shortcuts.")

Scenario 1
Observation:    Heatmap is concentrated on the animal body and head.
Interpretation: Good sign: the model uses relevant animal features.

Scenario 2
Observation:    Heatmap is mostly on grass or sky, not on the animal.
Interpretation: Possible shortcut: the model may rely on background context.

Scenario 3
Observation:    Heatmap covers only a corner of the crop.
Interpretation: The prediction may be unstable; inspect the crop and prediction confidence.

Defense sentence:
Grad-CAM is supporting interpretability evidence: it suggests whether the model focuses on animal features or on dataset shortcuts.


## 9. Hard-question simulator

Запускай `ask_hard_question()` и отвечай вслух. После ответа вызови `answer_hard_question(i)`.

In [22]:
HARD_QUESTIONS = [
    {
        "q": "If VGG16 is best, why did you need a custom CNN baseline?",
        "a": "The baseline is necessary to know whether transfer learning actually improves over a model trained from scratch. In this project, VGG16 improved over the baseline, but ResNet50 and MobileNetV2 did not.",
    },
    {
        "q": "Can you claim that pretrained models are always better?",
        "a": "No. The results are mixed: VGG16 is better, while ResNet50 and MobileNetV2 are below the custom CNN baseline in this setup.",
    },
    {
        "q": "Why not use all WAID classes?",
        "a": "The thesis task is livestock monitoring. Classes like zebra, seal, kiang, and camelus are outside the cattle/sheep farm-livestock scope.",
    },
    {
        "q": "What is the most important limitation?",
        "a": "The current implemented results are classification results on crops. Counting requires object detection on full images and separate mAP, MAE, and R² evaluation.",
    },
    {
        "q": "What would you do next?",
        "a": "Complete YOLOv8 detection/counting, compute mAP and count MAE/R², run subgroup evaluation by lighting and occlusion, and test fine-tuning of pretrained backbones.",
    },
]


def ask_hard_question():
    i = random.randrange(len(HARD_QUESTIONS))
    print(f"Hard question #{i}:")
    print(HARD_QUESTIONS[i]["q"])
    return i


def answer_hard_question(i):
    print("Reference answer:")
    print(HARD_QUESTIONS[i]["a"])

hard_i = ask_hard_question()

Hard question #0:
If VGG16 is best, why did you need a custom CNN baseline?


## 10. Final oral checklist

Перед защитой ты должна без подсказки проговорить:

- тему за 30 секунд;
- зачем нужны WAID и COCO;
- почему crops, а не full images для classification;
- почему image-level split защищает от leakage;
- что такое accuracy, precision, recall, F1;
- почему VGG16 лучшая именно в этом setup;
- почему ResNet50/MobileNetV2 не обязаны быть лучше baseline;
- что показал Grad-CAM;
- что уже реализовано, а что является планом для YOLOv8/counting;
- три ограничения работы и три направления future work.

Мини-правило: каждый ответ = прямой тезис + одна цифра + одно ограничение/trade-off.